<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/02_ML_Engineer/beginner/02_git_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Git Workflow for ML Teams

Good Git hygiene is as important for ML engineers as algorithmic knowledge. This notebook covers the workflows, branching strategies, and automation tools that professional teams use.

**Topics:** Branching strategies, commit conventions, pre-commit hooks, Git LFS for models, GitHub Actions basics

## 1. Git Branching Strategies for ML

In [ ]:
# This notebook documents git commands and workflows
# Run these in your terminal, not inside Python

print("""
=== Git Branching Strategy for ML Projects ===

Recommended: GitHub Flow (simpler than GitFlow)

Branch naming conventions:
  main              → production-ready code
  develop           → integration branch  
  feature/EXP-123-add-bert-embeddings  → experiments
  fix/fix-memory-leak-inference        → bug fixes
  release/v2.1.0    → release preparation
  hotfix/fix-nan-predictions           → urgent production fixes

Typical ML workflow:
  git checkout -b feature/EXP-456-ensemble-model
  
  # ... experiment ... run training ...
  
  git add src/ notebooks/ configs/
  git commit -m "feat(model): add gradient boosting ensemble\n\nAUC improved from 0.891 to 0.924 on validation set."
  
  git push origin feature/EXP-456-ensemble-model
  
  # Open PR, get code review, merge after CI passes
  git checkout main && git pull
  git branch -d feature/EXP-456-ensemble-model
""")

## 2. Conventional Commits for ML

In [ ]:
print("""
=== Conventional Commits Specification ===

Format: <type>[optional scope]: <description>
        [optional body]
        [optional footer(s)]

Types:
  feat     → new feature (correlates with MINOR version)
  fix      → bug fix (correlates with PATCH version)
  docs     → documentation only
  style    → formatting, no logic change
  refactor → code change, no bug fix or feature
  test     → adding tests
  chore    → build, CI, tooling changes
  perf     → performance improvement
  exp      → ML experiment (ML-specific extension)
  data     → data changes (ML-specific extension)

ML Examples:

  git commit -m "feat(model): add BERT embeddings to feature pipeline
  
  Replaces TF-IDF vectorizer with sentence-transformers all-MiniLM-L6-v2.
  AUC: 0.891 → 0.924 on dev set, latency: 12ms → 48ms.
  
  Closes #234"
  
  git commit -m "exp(EXP-456): add gradient boosting ensemble"
  
  git commit -m "data: add 50k new training examples from Q4 data pipeline"
  
  git commit -m "perf(inference): batch prediction to reduce latency 4x
  
  Changed single-item prediction to batch size 32.
  p50 latency: 48ms → 12ms, p99: 320ms → 95ms."
  
  git commit -m "fix(preprocessing): handle NaN in income feature
  
  Fixes production crash when income=NaN for new users.
  Now uses median imputation consistent with training.
  
  Fixes #891"
""")

## 3. Pre-commit Hooks

In [ ]:
# Pre-commit hooks run automatically before each commit
# Catch issues BEFORE they get into the codebase
# Install: pip install pre-commit && pre-commit install

precommit_config = """
# .pre-commit-config.yaml — place in repo root
repos:
  - repo: https://github.com/pre-commit/pre-commit-hooks
    rev: v4.5.0
    hooks:
      - id: trailing-whitespace
      - id: end-of-file-fixer
      - id: check-yaml
      - id: check-json
      - id: check-toml
      - id: check-merge-conflict
      - id: check-added-large-files
        args: ['--maxkb=1000']  # prevent accidentally committing large model files
      - id: detect-private-key           # prevent API key leaks!
      - id: check-ast                    # Python syntax check

  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.1.9
    hooks:
      - id: ruff              # fast Python linter (replaces flake8)
        args: [--fix]
      - id: ruff-format       # fast Python formatter (replaces black)

  - repo: https://github.com/pre-commit/mirrors-mypy
    rev: v1.8.0
    hooks:
      - id: mypy              # static type checking
        additional_dependencies: [types-requests]

  # Custom ML-specific hooks
  - repo: local
    hooks:
      - id: check-model-size
        name: Check model files are in .gitignore
        entry: python scripts/check_no_model_files.py
        language: python
        pass_filenames: false
      
      - id: run-unit-tests
        name: Run unit tests
        entry: pytest tests/unit/ -x -q
        language: system
        pass_filenames: false
        stages: [push]  # only on push, not every commit
"""

print('=== .pre-commit-config.yaml ===')
print(precommit_config)

print("""Commands:
  pre-commit install          → install hooks in .git/hooks/
  pre-commit run --all-files  → run on all files (first time)
  pre-commit autoupdate       → update hook versions
  git commit --no-verify      → skip hooks (emergency only!)
""")

## 4. Git LFS for Models and Data

In [ ]:
print("""
=== Git LFS (Large File Storage) for ML ===

Problem: Model weights (.pkl, .pt, .h5) can be 100MB-10GB
Solution: Git LFS stores large files outside the Git history,
          replaces them with lightweight pointers

Setup:
  brew install git-lfs         # or apt-get install git-lfs
  git lfs install              # initialize LFS for your user

Track file types:
  git lfs track "*.pkl"
  git lfs track "*.pt"
  git lfs track "*.h5"
  git lfs track "*.parquet"
  git lfs track "*.csv"
  git add .gitattributes

.gitattributes (auto-generated by git lfs track):
  *.pkl filter=lfs diff=lfs merge=lfs -text
  *.pt  filter=lfs diff=lfs merge=lfs -text
  *.h5  filter=lfs diff=lfs merge=lfs -text

Check LFS status:
  git lfs ls-files   → list tracked LFS files
  git lfs status     → show LFS files in staging area

⚠️  Better alternatives for ML:
  DVC (Data Version Control) — purpose-built for ML, integrates with S3/GCS
  MLflow artifacts            — attach artifacts to experiment runs
  Weights & Biases artifacts  — versioned model registry
  HuggingFace Hub             — model hub with versioning

Recommended .gitignore for ML projects:
  # Models (use DVC or MLflow instead)
  *.pkl
  *.pt
  *.h5
  models/
  
  # Data (use DVC instead)
  data/raw/
  data/processed/
  *.parquet
  *.csv
  
  # Experiment outputs
  mlruns/
  outputs/
  wandb/
  
  # Credentials (NEVER commit these)
  .env
  *.key
  secrets.yaml
""")

## 5. GitHub Actions for ML CI/CD

In [ ]:
github_actions_workflow = """
# .github/workflows/ml-ci.yml
name: ML CI Pipeline

on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
          cache: 'pip'
      
      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install -r requirements-dev.txt
      
      - name: Lint (ruff)
        run: ruff check src/ tests/
      
      - name: Type check (mypy)
        run: mypy src/
      
      - name: Unit tests
        run: pytest tests/unit/ -v --cov=src/ --cov-report=xml
      
      - name: Upload coverage
        uses: codecov/codecov-action@v3
  
  model-eval:
    needs: test
    runs-on: ubuntu-latest
    if: github.event_name == 'pull_request'
    
    steps:
      - uses: actions/checkout@v4
      
      - name: Train model on subset
        run: python scripts/train.py --fast-mode --sample-size 1000
      
      - name: Evaluate model
        run: python scripts/evaluate.py --threshold 0.90
        # Fails if AUC drops below threshold
      
      - name: Comment PR with results
        uses: actions/github-script@v7
        with:
          script: |
            const results = require('./eval_results.json');
            github.rest.issues.createComment({
              issue_number: context.issue.number,
              owner: context.repo.owner,
              repo: context.repo.repo,
              body: `## Model Evaluation Results\n\nAUC: ${results.auc}\nF1: ${results.f1}`
            });
"""

print('=== GitHub Actions ML CI Pipeline ===')
print(github_actions_workflow)

# Git workflow summary
print('\n=== ML Team Git Workflow Summary ===')
print('''
Daily workflow:
  1. git checkout -b feature/EXP-{ticket}-{description}
  2. Experiment, commit often with meaningful messages
  3. Pre-commit hooks catch style/type issues before commit
  4. Push branch, open PR
  5. CI runs: lint + type check + unit tests + model eval
  6. Code review: algorithm correctness, edge cases, efficiency
  7. Merge to main when CI passes + review approved
  8. Delete feature branch

Golden rules:
  • Never commit secrets (.env, API keys)
  • Never commit large files (models, data) — use DVC/LFS
  • Never force-push to main/develop
  • Always write tests for preprocessing logic
  • Keep PRs small (<400 lines) for reviewability
''')